# CommGuard calibration v3

Canonical dual-T4 calibration source. It records environment and bounded collective observations; it makes no detector claim. Run with Internet enabled only for the immutable Git fetch, and never add credentials to this notebook.

> **Prototype scope:** CommGuard’s Kaggle workflow is a single-node, dual-NVIDIA-T4 research prototype. It validates experimental methodology and software behavior on two local GPU ranks. It does not establish generalization to two physical 8-GPU nodes, NVLink/NVSwitch fabrics, RoCE or InfiniBand networks, large frontier-model workloads, or production treaty-verification deployments.


In [ ]:
import hashlib
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_calibration_v3"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
INSTALL_SOURCE = "auto"  # auto: wheel, source archive, pinned Git commit, then dev source.
PINNED_PUBLIC_COMMIT = ""  # Required for public Git installation.
EXPECTED_PACKAGE_SHA256 = ""  # Required for a supplied wheel or source archive.
EXPECTED_NOTEBOOK_SHA256 = ""  # SHA-256 of this canonical source notebook.
DEVELOPMENT_SMOKE_TEST = False
DEVELOPMENT_SOURCE = Path("/kaggle/working/commguard-development-source")
REPOSITORY = Path("/kaggle/working/commguard-source")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

wheel_candidates = sorted(Path("/kaggle/input").rglob("commguard*.whl"))
archive_candidates = sorted(
    path for path in Path("/kaggle/input").rglob("commguard*")
    if path.is_file() and path.name.endswith((".tar.gz", ".zip"))
    and not any(token in path.name for token in (
        "-prototype-", "review-bundle", "calibration", "benign-corpus",
        "detector-evaluation", "adversarial-redteam",
    ))
)
selected = None
method = INSTALL_SOURCE
if method == "auto":
    method = "wheel" if wheel_candidates else "archive" if archive_candidates else "git"
if method == "wheel":
    if len(wheel_candidates) != 1:
        raise RuntimeError(f"Expected exactly one CommGuard wheel, observed {wheel_candidates}")
    selected = wheel_candidates[0]
elif method == "archive":
    if len(archive_candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one CommGuard source archive, observed {archive_candidates}"
        )
    selected = archive_candidates[0]

if selected is not None:
    actual_package_sha256 = file_sha256(selected)
    if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_PACKAGE_SHA256):
        raise RuntimeError("Set EXPECTED_PACKAGE_SHA256 for the supplied package.")
    if actual_package_sha256 != EXPECTED_PACKAGE_SHA256:
        raise RuntimeError("Supplied package SHA-256 does not match.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(selected)], check=True
    )
    SOURCE_IDENTITY = f"sha256:{actual_package_sha256}"
    SOURCE_DIRTY = False
    INSTALL_PROVENANCE = {
        "install_source": method,
        "install_path": str(selected),
        "package_sha256": actual_package_sha256,
        "source_identity": SOURCE_IDENTITY,
    }
elif method == "git":
    if not re.fullmatch(r"[0-9a-f]{40}", PINNED_PUBLIC_COMMIT):
        raise RuntimeError("Set PINNED_PUBLIC_COMMIT to a pushed 40-character commit.")
    if not REPOSITORY.exists():
        subprocess.run(
            [
                "git", "clone", "--filter=blob:none", "--no-checkout",
                REPOSITORY_URL, str(REPOSITORY),
            ],
            check=True,
        )
    if not (REPOSITORY / ".git").is_dir():
        raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "fetch", "origin", PINNED_PUBLIC_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "checkout", "--detach", PINNED_PUBLIC_COMMIT], check=True
    )
    head = subprocess.run(
        ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    dirty = subprocess.run(
        ["git", "-C", str(REPOSITORY), "status", "--porcelain"], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    pushed_refs = subprocess.run(
        ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    if head != PINNED_PUBLIC_COMMIT or dirty or not pushed_refs:
        raise RuntimeError("Pinned Git source is dirty, mismatched, or not remote-visible.")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-build-isolation",
            "--no-deps", str(REPOSITORY),
        ],
        check=True,
    )
    SOURCE_IDENTITY = head
    SOURCE_DIRTY = False
    INSTALL_PROVENANCE = {
        "install_source": "pinned_public_git_commit",
        "repository_url": REPOSITORY_URL,
        "source_identity": head,
        "remote_refs": pushed_refs.splitlines(),
    }
elif method == "development":
    if not DEVELOPMENT_SMOKE_TEST or not (DEVELOPMENT_SOURCE / "pyproject.toml").is_file():
        raise RuntimeError(
            "Editable development source is allowed only for an explicit smoke test."
        )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-build-isolation",
            "--no-deps", "-e", str(DEVELOPMENT_SOURCE),
        ],
        check=True,
    )
    SOURCE_IDENTITY = "development-editable"
    SOURCE_DIRTY = True
    INSTALL_PROVENANCE = {
        "install_source": "editable_local_development",
        "source_identity": SOURCE_IDENTITY,
        "development_smoke_only": True,
    }
else:
    raise RuntimeError(f"Unsupported INSTALL_SOURCE={method!r}")

importlib.invalidate_caches()
for module_name in [
    name for name in sys.modules if name == "commguard" or name.startswith("commguard.")
]:
    del sys.modules[module_name]
import commguard
REVIEWED_COMMIT = SOURCE_IDENTITY
PIP_FREEZE = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True
).stdout.splitlines()
INSTALL_PROVENANCE.update({
    "commguard_version": commguard.__version__,
    "commguard_import": str(Path(commguard.__file__).resolve()),
    "python_version": sys.version,
    "pip_freeze": PIP_FREEZE,
})
print({
    "source_identity": SOURCE_IDENTITY,
    "source_dirty": SOURCE_DIRTY,
    "installation": INSTALL_PROVENANCE,
})


In [ ]:
import shutil

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("nvidia-smi is required for the strict dual-T4 calibration.")
gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,compute_cap",
        "--format=csv,noheader,nounits",
    ],
    check=True,
    capture_output=True,
    text=True,
)
GPU_SUMMARY = []
for line in gpu_query.stdout.splitlines():
    index, name, memory_mib, compute_capability = [part.strip() for part in line.split(",")]
    GPU_SUMMARY.append({
        "index": int(index),
        "name": name,
        "memory_total_mib": int(memory_mib),
        "compute_capability": compute_capability,
    })
if len(GPU_SUMMARY) != 2 or any("T4" not in gpu["name"] for gpu in GPU_SUMMARY):
    raise RuntimeError(f"Expected exactly two Tesla T4 GPUs, observed: {GPU_SUMMARY}")
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("PyTorch must expose exactly two CUDA devices.")
if not torch.distributed.is_nccl_available():
    raise RuntimeError("The reviewed PyTorch build does not expose NCCL.")
print({
    "gpus": GPU_SUMMARY,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "nccl_available": torch.distributed.is_nccl_available(),
})


In [ ]:
from datetime import datetime, timezone

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ARTIFACTS = Path(f"/kaggle/working/commguard-calibration-v3-{NOTEBOOK_RUN_ID}")
if ARTIFACTS.exists():
    raise RuntimeError(
        f"Refusing reused calibration workspace {ARTIFACTS}; create a new NOTEBOOK_RUN_ID."
    )
ARTIFACTS.mkdir(parents=True, exist_ok=False)
write_probe = ARTIFACTS / ".write-probe"
with write_probe.open("x", encoding="utf-8") as stream:
    stream.write("writable\n")
write_probe.unlink()
print({"notebook_run_id": NOTEBOOK_RUN_ID, "artifact_workspace": ARTIFACTS.name})


In [ ]:
from commguard.artifacts import ArtifactStore
from commguard.schemas import CURRENT_SCHEMA_VERSION, SCHEMA_VERSION

ArtifactStore(ARTIFACTS).initialize()
BOOTSTRAP = {
    "artifact_kind": "experiment_summary",
    "schema_version": SCHEMA_VERSION,
    "summary_type": "notebook_bootstrap",
    "notebook_version": NOTEBOOK_VERSION,
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "reviewed_commit": REVIEWED_COMMIT,
    "source_repository": "commguard-source",
    "gpu_summary": GPU_SUMMARY,
    "artifact_schema_version": CURRENT_SCHEMA_VERSION,
}
ArtifactStore(ARTIFACTS).write_json("environment/notebook-bootstrap.json", BOOTSTRAP)
print(BOOTSTRAP)


In [ ]:


import socket
from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

NOTEBOOK_FILENAME = f"{NOTEBOOK_VERSION}.ipynb"
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_NOTEBOOK_SHA256):
    raise RuntimeError("Set EXPECTED_NOTEBOOK_SHA256 to the canonical notebook source hash.")
if (REPOSITORY / "notebooks" / NOTEBOOK_FILENAME).is_file():
    actual_notebook_sha256 = file_sha256(REPOSITORY / "notebooks" / NOTEBOOK_FILENAME)
    if actual_notebook_sha256 != EXPECTED_NOTEBOOK_SHA256:
        raise RuntimeError("Canonical notebook SHA-256 does not match the pinned Git source.")
DIRTY_SOURCE_SMOKE_ONLY = bool(SOURCE_DIRTY and DEVELOPMENT_SMOKE_TEST)
if SOURCE_DIRTY and not DIRTY_SOURCE_SMOKE_ONLY:
    raise RuntimeError("Dirty source cannot create accepted research evidence.")
CONTEXT = ProvenanceContext(
    corpus_id=f"corpus-calibration-v3-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-calibration-v3-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-calibration-v3-{NOTEBOOK_RUN_ID}",
    node_id=socket.gethostname(),
    source_commit=SOURCE_IDENTITY,
    source_dirty=SOURCE_DIRTY,
    notebook_version="commguard_calibration_v3",
    input_archive_sha256=None,
    random_seed=20260730,
)
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "source_dirty": CONTEXT.source_dirty,
    "notebook_sha256": EXPECTED_NOTEBOOK_SHA256,
    "development_smoke_only": DIRTY_SOURCE_SMOKE_ONLY,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
from commguard.orchestrator import run_calibration_sweep
from commguard.telemetry import compare_sampling_intervals

RUN_MODE = "smoke"  # "smoke" or "full"
if RUN_MODE not in {"smoke", "full"}:
    raise RuntimeError("RUN_MODE must be smoke or full.")
DEVELOPMENT_SMOKE_ONLY = RUN_MODE == "smoke"
if SOURCE_DIRTY and RUN_MODE != "smoke":
    raise RuntimeError("Dirty editable source is restricted to RUN_MODE='smoke'.")
DEVELOPMENT_SMOKE_ONLY = DEVELOPMENT_SMOKE_ONLY or DIRTY_SOURCE_SMOKE_ONLY
PAYLOADS = (16,) if DEVELOPMENT_SMOKE_ONLY else (1, 4, 16, 64, 128)
REPETITIONS = 1 if DEVELOPMENT_SMOKE_ONLY else 5
SAMPLING_INTERVAL_S = 0.5
SAMPLING_COMPARISON = compare_sampling_intervals(
    f"sampling-{NOTEBOOK_RUN_ID}",
    intervals_s=(1.0, 0.5, 0.2),
    duration_s=3.0 if DEVELOPMENT_SMOKE_ONLY else 10.0,
)
print({
    "run_mode": RUN_MODE,
    "development_smoke_only": DEVELOPMENT_SMOKE_ONLY,
    "scientific_acceptance_eligible": not DEVELOPMENT_SMOKE_ONLY,
    "idle_repetitions": REPETITIONS,
    "collective": "all_reduce",
    "payload_mib": list(PAYLOADS),
    "total_runs": REPETITIONS * (1 + len(PAYLOADS)),
})
CALIBRATION = run_calibration_sweep(
    output=ARTIFACTS,
    payload_mib=PAYLOADS,
    repetitions=REPETITIONS,
    sampling_interval_s=SAMPLING_INTERVAL_S,
    timeout_s=180.0,
    provenance=CONTEXT,
    progress_callback=lambda event: print({"calibration_progress": event}),
    development_smoke_only=DEVELOPMENT_SMOKE_ONLY,
)
print({
    "status": CALIBRATION["status"],
    "result_state": CALIBRATION["result_state"],
    "decision_state": CALIBRATION["decision_state"],
    "idle_usable_repetitions": CALIBRATION["idle_baseline_usable_repetitions"],
    "payload_summaries": CALIBRATION["payload_summaries"],
    "standard_sweep_validation": CALIBRATION["standard_sweep_validation"],
    "exact_calibration_reference_for_next_notebook": CALIBRATION["reference"],
})


In [ ]:
from commguard.artifacts import materialize_calibration_package

CALIBRATION_PACKAGE = materialize_calibration_package(
    ARTIFACTS,
    CALIBRATION,
    SAMPLING_COMPARISON,
    environment=ENVIRONMENT,
    provenance={**CONTEXT.to_dict(), **INSTALL_PROVENANCE},
    notebook_filename=NOTEBOOK_FILENAME,
    notebook_sha256=EXPECTED_NOTEBOOK_SHA256,
    configuration={
        "run_mode": RUN_MODE,
        "payload_mib": list(PAYLOADS),
        "repetitions": REPETITIONS,
        "sampling_interval_s": SAMPLING_INTERVAL_S,
    },
    development_smoke_only=DEVELOPMENT_SMOKE_ONLY,
)
print({"machine_readable_package": str(CALIBRATION_PACKAGE)})


## Results

not executed. The committed notebook contains no runtime result or output.


In [ ]:
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-calibration-prototype-{NOTEBOOK_RUN_ID}.tar.gz")
ARCHIVE, SHA_FILE = ArtifactStore(ARTIFACTS).export_with_checksum(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
print({
    "archive": str(ARCHIVE),
    "archive_sha256": ARCHIVE_SHA256,
    "sha256_file": str(SHA_FILE),
    "calibration_status": CALIBRATION["status"],
    "calibration_artifact_reference": CALIBRATION["reference"],
})
if DEVELOPMENT_SMOKE_ONLY:
    print("DEVELOPMENT SMOKE ONLY: run full mode in a new workspace before benign collection.")
elif CALIBRATION["result_state"] != "supported" or not CALIBRATION["modern_capture_gate_passed"]:
    print("FAILED/INCONCLUSIVE EVIDENCE WAS PRESERVED. Do not run the benign notebook.")
    raise RuntimeError(
        "Modern idle-aware calibration is not supported; download the diagnostic archive "
        "and SHA-256 file, then investigate before creating a fresh NOTEBOOK_RUN_ID."
    )
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(
    "NEXT STEP: copy the exact archive SHA-256 and calibration artifact reference into "
    "commguard_benign_corpus_v2.ipynb."
)
